# [AI] ROTBOTS -- Script Generator
## From Sources to Storyboard

---

1. **Feed the Machine** -- Paste URLs or text, scrape and summarize
2. **The Script Writer** -- LLM generates essay with narration + visual directions
3. **Visual Translation** -- Storyboard with styles and T2V prompts

**You don't need to understand the code.** Just fill in inputs and press > Play.

---

In [ ]:
# ============================================================
# SETUP
# ============================================================
!pip install -q requests beautifulsoup4 pymupdf

import json, re, random, requests, time as _time
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import display, Markdown, HTML

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
BASE_DIR.mkdir(parents=True, exist_ok=True)
print('[OK] Setup complete')

In [ ]:
# ============================================================
# API KEY (starts with gsk_)
# Get free: https://console.groq.com/keys
# ============================================================
GROQ_API_KEY = ''  # <-- Paste key here

LLM_MODEL = 'llama-3.3-70b-versatile'
LLM_TEMPERATURE = 0.8
LLM_MAX_TOKENS = 4096
GROQ_API_URL = 'https://api.groq.com/openai/v1/chat/completions'  # NOT here

if not GROQ_API_KEY: print('[!]  Paste your Groq API key above!')
elif not GROQ_API_KEY.startswith('gsk_'): print('[!]  Key should start with gsk_')
else: print(f'[OK] API key configured ({LLM_MODEL})')

In [ ]:
# LOAD SESSION FROM VIDEO PLAN
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'video_plan.json').exists()])
if not sessions: print('[!!] No plan! Run 01_Video_Plan first.')
else:
    for i,s in enumerate(sessions): print(f'   {i}: {s}')
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
with open(SESSION_DIR/'video_plan.json') as f: plan = json.load(f)

TOPIC = plan['topic']
SOURCES = plan['sources']
SECONDS_PER_SCENE = plan['seconds_per_scene']
NUM_TOTAL_SCENES = plan.get('num_total_scenes', plan.get('total_content_scenes', 10))
NUM_CHAPTERS = plan.get('num_chapters', max(1, min(3, NUM_TOTAL_SCENES // 3)))
SECTIONS_PER_CHAPTER = plan.get('sections_per_chapter', max(1, NUM_TOTAL_SCENES // NUM_CHAPTERS))
WORDS_PER_SCENE = plan.get('words_per_scene', SECONDS_PER_SCENE * 2.5)
scene_types = plan.get('scene_types', ['ai_generated'] * NUM_TOTAL_SCENES)

print(f'[OK] Session: {SESSION_NAME}')
print(f'{NUM_TOTAL_SCENES} scenes, {NUM_CHAPTERS} chapters, ~{int(WORDS_PER_SCENE)} words/scene')
print(f'Scene types: {scene_types}')
TARGET_VIDEO_LENGTH = plan.get('total_video_length', NUM_TOTAL_SCENES * SECONDS_PER_SCENE)
TOTAL_NARRATION_WORDS = int(plan.get('content_length', TARGET_VIDEO_LENGTH) * 2.5)
print(f'Narration target: {TOTAL_NARRATION_WORDS} words for ~{plan.get("content_length", TARGET_VIDEO_LENGTH):.0f}s')


In [ ]:
# HELPER FUNCTIONS
def progress_bar(current, total, label='', width=30):
    pct = current / total if total > 0 else 0
    filled = int(width * pct)
    bar = '#' * filled + '-' * (width - filled)
    return f'<div style="font-family:monospace;font-size:14px;padding:2px 0;">{bar} {current}/{total} {label} ({pct:.0%})</div>'

def progress_html(title, current, total, label='', detail='', color='#4ecca3'):
    return (
        f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;">' +
        f'<div style="font-size:16px;font-weight:bold;color:{color};">{title}</div>' +
        progress_bar(current, total, label) +
        (f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">{detail}</div>' if detail else '') +
        '</div>'
    )

def query_llm(prompt, system_prompt=None, temperature=None):
    messages = []
    if system_prompt: messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})
    payload = {'model': LLM_MODEL, 'messages': messages, 'temperature': temperature or LLM_TEMPERATURE, 'max_tokens': LLM_MAX_TOKENS}
    response = requests.post(GROQ_API_URL, headers={'Authorization': f'Bearer {GROQ_API_KEY}', 'Content-Type': 'application/json'}, json=payload, timeout=120)
    response.raise_for_status()
    content = response.json()['choices'][0]['message']['content']
    if '<think>' in content and '</think>' in content: content = content.split('</think>')[-1].strip()
    return content

def parse_json_response(response):
    response = response.strip()
    response = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', response)
    if response.startswith('```'):
        lines = response.split('\n')
        response = '\n'.join(lines[1:-1] if lines[-1].strip() == '```' else lines[1:])
    response = response.strip()
    if not response.startswith('[') and not response.startswith('{'):
        for ch in ['{', '[']:
            idx = response.find(ch)
            if idx != -1: response = response[idx:]; break
    for end_char in ['}', ']']:
        last_idx = response.rfind(end_char)
        if last_idx != -1:
            truncated = response[:last_idx + 1]
            for text in [truncated, re.sub(r',\s*([}\]])', r'\1', truncated)]:
                try: return json.loads(text, strict=False)
                except json.JSONDecodeError: pass
    return json.loads(re.sub(r',\s*([}\]])', r'\1', response), strict=False)

def fetch_website_text(url, max_chars=10000):
    response = requests.get(url.strip(), headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    for el in soup(['script','style','nav','header','footer','aside','form']): el.decompose()
    main = None
    for sel in ['article','main','[role="main"]','.content','#content']:
        main = soup.select_one(sel)
        if main: break
    text = main.get_text(separator=' ',strip=True) if main else (soup.find('body') or soup).get_text(separator=' ',strip=True)
    return re.sub(r'\s+', ' ', text).strip()[:max_chars]

def fetch_pdf_text(source, max_chars=10000):
    import tempfile
    try: import pymupdf as fitz
    except ImportError: import fitz
    temp_file = None
    try:
        if source.startswith('http'):
            resp = requests.get(source, headers={'User-Agent':'Mozilla/5.0'}, timeout=60); resp.raise_for_status()
            temp_file = tempfile.NamedTemporaryFile(suffix='.pdf', delete=False); temp_file.write(resp.content); temp_file.close()
            pdf_path = temp_file.name
        else: pdf_path = source
        doc = fitz.open(pdf_path); parts = []; total = 0
        for page in doc: t = page.get_text(); parts.append(t); total += len(t);
        doc.close()
        return re.sub(r'\n{3,}','\n\n','\n'.join(parts))[:max_chars]
    finally:
        if temp_file: import os; os.unlink(temp_file.name)

print('[OK] Helpers loaded')

---
## [DL] Station 1: Feed the Machine

In [ ]:
# SCRAPE + SUMMARIZE with live progress
prog = display(HTML(''), display_id=True)
source_texts = []
total_steps = len(SOURCES) * 2  # scrape + summarize each
step = 0
t0 = _time.time()

for i, source in enumerate(SOURCES):
    prog.update(HTML(progress_html(f'[DL] Scraping source {i+1}/{len(SOURCES)}', step, total_steps, 'steps', source[:60])))
    try:
        if source.startswith('http') and source.lower().endswith('.pdf'): text = fetch_pdf_text(source)
        elif source.startswith('http'): text = fetch_website_text(source)
        else: text = source
        source_texts.append({'source': source[:100], 'text': text})
    except Exception as e: pass
    step += 1

summaries = []
for i, src in enumerate(source_texts):
    prog.update(HTML(progress_html(f'[BRAIN] Summarizing source {i+1}/{len(source_texts)}', step, total_steps, 'steps', f'LLM call... ({src["source"][:40]})')))
    summary = query_llm(f'Summarize in 2-3 short paragraphs for a video essay about "{TOPIC}":\n\n{src["text"][:6000]}', system_prompt='Be concise.')
    summaries.append({'source': src['source'], 'summary': summary})
    step += 1

with open(SESSION_DIR / 'summaries.json', 'w') as f: json.dump({'topic': TOPIC, 'sources': summaries}, f, indent=2)
prog.update(HTML(progress_html(f'[OK] {len(summaries)} sources scraped & summarized in {_time.time()-t0:.1f}s', total_steps, total_steps, 'steps')))

In [ ]:
# VIEW
for i, s in enumerate(summaries):
    display(Markdown(f'### Source {i+1}: {s["source"]}'))
    display(Markdown(s['summary'] + '\n\n---'))

---
## >> Station 2: The Script Writer

In [ ]:
# OUTLINE + ESSAY NARRATION
total_llm = 3 + NUM_CHAPTERS
prog = display(HTML(''), display_id=True)
t0 = _time.time()

# Step 1: Outline
prog.update(HTML(progress_html('Generating outline...', 0, total_llm, 'LLM calls')))
summaries_text = '\n\n'.join([f'SOURCE: {s["source"]}\n{s["summary"]}' for s in summaries])
outline_prompt = f'Create an essay outline for a {TARGET_VIDEO_LENGTH}s video about: "{TOPIC}"\n\nRESEARCH:\n{summaries_text}\n\nExactly {NUM_CHAPTERS} chapters. JSON only:\n{{"title": "...", "thesis": "...", "chapters": [{{"chapter": 1, "title": "...", "summary": "...", "key_points": ["..."]}}]}}'
for attempt in range(3):
    try: outline = parse_json_response(query_llm(outline_prompt, 'Expert essay writer. Concise.')); break
    except Exception as e: prog.update(HTML(progress_html(f'Retry {attempt+1}/3...', 0, total_llm, 'LLM calls', str(e)[:60], '#f39c12')))
if len(outline.get('chapters',[])) > NUM_CHAPTERS: outline['chapters'] = outline['chapters'][:NUM_CHAPTERS]

# Step 2: Full essay narration
prog.update(HTML(progress_html('Writing full essay narration...', 1, total_llm, 'LLM calls')))
chapter_summaries = '\n'.join([f'Ch {c["chapter"]}: {c["title"]} - {c.get("summary","")}' for c in outline['chapters']])
essay_prompt = f'''Write a single continuous narration for a {TARGET_VIDEO_LENGTH}-second video about: "{TOPIC}"

STRUCTURE:
{chapter_summaries}

RULES:
- Write EXACTLY {TOTAL_NARRATION_WORDS} words total
- One flowing essay, NO section breaks or headers
- Engaging, documentary-style prose
- Write complete sentences, not fragments
- Output ONLY the essay text, nothing else'''
essay_narration = query_llm(essay_prompt, f'Expert video essay scriptwriter. Write exactly {TOTAL_NARRATION_WORDS} words of flowing narration.').strip()
wc = len(essay_narration.split())
while wc < int(TOTAL_NARRATION_WORDS * 0.9):
    needed = TOTAL_NARRATION_WORDS - wc
    print(f'  {wc} words -- expanding by {needed}...')
    more = query_llm(f'Continue this essay with EXACTLY {needed} more words. Same style, same topic:\n\n{essay_narration[-500:]}', f'Continue seamlessly. Write exactly {needed} words.').strip()
    essay_narration = essay_narration + ' ' + more
    wc = len(essay_narration.split())

# Step 3: Visual directions per chapter
for i, ch in enumerate(outline['chapters']):
    prog.update(HTML(progress_html(f'Visual directions: Ch {ch.get("chapter",i+1)}', 2+i, total_llm, 'LLM calls')))
    vis_prompt = f'Write {SECTIONS_PER_CHAPTER} visual scene descriptions for chapter "{ch["title"]}" of a video about "{TOPIC}". JSON: [{{"section": 1, "visual_direction": "describe the visual", "mood": "emotional tone"}}]'
    try:
        sections = parse_json_response(query_llm(vis_prompt, 'Visual director. Concise scene descriptions.'))
        if isinstance(sections, dict):
            for v in sections.values():
                if isinstance(v, list): sections = v; break
            else: sections = [sections]
    except: sections = [{'section': 1, 'visual_direction': ch.get('summary',''), 'mood': 'neutral'}]
    outline['chapters'][i]['sections'] = sections[:SECTIONS_PER_CHAPTER]

# Save
outline['narration'] = essay_narration
outline['credits'] = {'sources': [s['source'] for s in summaries]}
with open(SESSION_DIR / 'essay_script.json', 'w') as f: json.dump(outline, f, indent=2)
wc = len(essay_narration.split())
prog.update(HTML(progress_html(f'"{outline.get("title","")}" -- {wc} words ~ {wc/2.5:.0f}s ({_time.time()-t0:.1f}s)', total_llm, total_llm, 'LLM calls')))


In [ ]:
# VIEW
display(Markdown(f'# {outline.get("title","")}\n*{outline.get("thesis","")}*\n\n---'))
display(Markdown(f'## Full Narration ({len(essay_narration.split())} words)\n\n> {essay_narration}\n\n---'))
for ch in outline['chapters']:
    display(Markdown(f'### Ch {ch["chapter"]}: {ch["title"]}'))
    for s in ch.get('sections',[]):
        display(Markdown(f'**Sec {s.get("section","?")}** *{s.get("mood","?")}* -- {s.get("visual_direction","")}'))


---
## [SCENE] Station 3: Visual Translation

In [ ]:
# STYLE ARC
STYLE_ARCS = {
    'calm_to_dramatic': {'description': 'Calm -> intense', 'early': ['documentary','nature'], 'middle': ['news_report','sports_commentary'], 'late': ['action_movie','horror']},
    'documentary_journey': {'description': 'Documentary, increasing energy', 'early': ['documentary'], 'middle': ['nature','news_report'], 'late': ['action_movie','sports_commentary']},
    'chaos_arc': {'description': 'Brainrot chaos', 'early': ['classic_brainrot','zoomer_brainrot'], 'middle': ['surreal_brainrot','hyperpop_brainrot'], 'late': ['fever_dream_brainrot','cursed_brainrot']},
}
STYLES = {
    'documentary':{'vk':'cinematic, professional lighting','ak':'ambient sounds'},
    'nature':{'vk':'wide nature shots, golden hour','ak':'nature sounds, wind'},
    'news_report':{'vk':'news studio, professional','ak':'news audio, serious'},
    'sports_commentary':{'vk':'dynamic angles, slow motion','ak':'crowd, energetic'},
    'action_movie':{'vk':'dynamic movement, dramatic lighting','ak':'orchestral hits'},
    'horror':{'vk':'dark lighting, shadows, fog','ak':'creepy ambience'},
    'classic_brainrot':{'vk':'chaotic, deep fried','ak':'bass boosted'},
    'surreal_brainrot':{'vk':'dreamlike, impossible geometry','ak':'slowed reverb'},
    'zoomer_brainrot':{'vk':'meme aesthetic, ironic','ak':'TikTok sounds'},
    'hyperpop_brainrot':{'vk':'maximalist, Y2K','ak':'hyperpop beats'},
    'fever_dream_brainrot':{'vk':'hallucinatory, warped','ak':'echoing'},
    'cursed_brainrot':{'vk':'unsettling, jpeg artifacts','ak':'distorted'},
}

CHOSEN_ARC = 'calm_to_dramatic'
arc = STYLE_ARCS[CHOSEN_ARC]
print(f'[ART] {CHOSEN_ARC}: {arc["description"]}')

In [ ]:
# STORYBOARD + STYLES
all_sec = []
for ch in outline.get('chapters', []):
    for sec in ch.get('sections', []):
        d = dict(**sec)
        d['chapter_title'] = ch['title']
        d['chapter'] = ch.get('chapter', 0)
        all_sec.append(d)

scenes = []
sn = 1; si = 0
for stype in scene_types:
    if si < len(all_sec):
        sec = all_sec[si]
    else:
        sec = dict(narration='', visual_direction='', mood='neutral', chapter_title=TOPIC, section=si+1)
    scenes.append(dict(
        scene=sn, scene_type=stype,
        title=str(sec.get('chapter_title','')) + ' - Part ' + str(sec.get('section', si+1)),
        visual_direction=sec.get('visual_direction', ''),
        mood=sec.get('mood', ''),
        duration=SECONDS_PER_SCENE))
    sn += 1; si += 1

ai_sc = [s for s in scenes if s['scene_type'] == 'ai_generated']
total_ai = len(ai_sc)
ee = max(1, int(total_ai * 0.25))
ls = max(ee + 1, int(total_ai * 0.75))
last = None
for i, sc in enumerate(ai_sc):
    phase = 'early' if i < ee else ('late' if i >= ls else 'middle')
    pool = arc.get(phase, ['documentary'])
    avail = [s for s in pool if s != last] or pool
    st = random.choice(avail)
    sc['assigned_style'] = st
    sc['visual_keywords'] = STYLES.get(st, {}).get('vk', '') if isinstance(STYLES.get(st), dict) else STYLES.get(st, '')
    last = st

with open(SESSION_DIR / 'storyboard.json', 'w') as f: json.dump(scenes, f, indent=2)
for s in scenes:
    tag = f' [{s.get("assigned_style","")}]' if s.get('assigned_style') else ''
    icon = {'ai_generated':'[AI]','archive':'[ARCH]','upload':'[UP]'}.get(s['scene_type'],'*')
    print(f'   {icon} Scene {s["scene"]}: [{s["scene_type"]}] {s["title"]}{tag}')

In [ ]:
# T2V PROMPTS (AI scenes only)
ai_scenes_list = [sc for sc in scenes if sc['scene_type'] == 'ai_generated']
if not ai_scenes_list:
    print(' No AI scenes in plan! Check ARCHIVE_RATIO / UPLOAD_RATIO.')
OPENINGS = ['Start with SHOT TYPE','Start with ACTION','Start with ENVIRONMENT','Start with LIGHTING','Start with CAMERA']
prompts = []
prog = display(HTML(''), display_id=True)
t0 = _time.time()

for idx, sc in enumerate(ai_scenes_list):
    st = sc.get('assigned_style','documentary')
    vk = sc.get('visual_keywords','')
    prog.update(HTML(progress_html(f'Scene {sc["scene"]}: {sc["title"]}', idx, len(ai_scenes_list), 'prompts', f'Style: {st}')))
    sys_p = f'T2V prompt expert. ONE paragraph, 3-5 sentences for {SECONDS_PER_SCENE}s clip. Style: {st}. Visual: {vk}. No text overlays.'
    user_p = f'T2V prompt for {SECONDS_PER_SCENE}s:\nVisual: {sc.get("visual_direction","")}\nMood: {sc.get("mood","")}\n{random.choice(OPENINGS)}\nOutput ONLY the prompt text.'
    t2v = query_llm(user_p, sys_p).strip().strip('"')
    prompts.append(dict(scene=sc['scene'], title=sc['title'], t2v_prompt=t2v,

with open(SESSION_DIR / 'prompts.json', 'w') as f: json.dump(prompts, f, indent=2)
prog.update(HTML(progress_html(f'[OK] {len(prompts)} T2V prompts in {_time.time()-t0:.1f}s', len(ai_scenes_list), len(ai_scenes_list), 'prompts')))

In [ ]:
# VIEW
display(Markdown(f'# [SCENE] {len(prompts)} scenes × ~{SECONDS_PER_SCENE}s = ~{len(prompts)*SECONDS_PER_SCENE}s\n\n---'))
for p in prompts:
    wc=len(p['narration'].split())
    display(Markdown(f'### Scene {p["scene"]}: {p["title"]}\n`{p["style"]}` | {p["duration"]}s | {wc}w\n\n> [MIC] {p["narration"]}\n\n> [?] {p["t2v_prompt"]}\n\n---'))
print(f'[DIR] Session "{SESSION_NAME}" saved to Google Drive')

---
*ROTBOTS Workshop -- LI-MA 2026*